# Orçamento de contexto para RAG

Selecione trechos por relevância, limite cada fonte e confira o tamanho do JSON que será enviado.

[Artigo completo](../article/ARTICLE.md)

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

cwd = Path.cwd()
lab = next(
    path
    for path in (cwd, cwd.parent, cwd / "0013_orcamento_contexto")
    if path.name == "0013_orcamento_contexto" and (path / "src" / "main.py").is_file()
)

def run(*options):
    completed = subprocess.run(
        [sys.executable, "-X", "utf8", str(lab / "src" / "main.py"), *options],
        capture_output=True, check=True, text=True, encoding="utf-8",
    )
    return json.loads(completed.stdout)

Execute a próxima célula para reproduzir a comparação e verificar os resultados com asserções.

Com 245 bytes, o campo `context` serializado ocupa 214 bytes e carrega dois trechos. Confira `skipped` para distinguir a cota de fonte do limite de bytes. Com 120 bytes, só `refund-summary` cabe.

In [ ]:
report = run("--bytes", "245", "--per-source", "1")
assert len(report["context"].encode("utf-8")) == report["bytes_used"] == 214
assert [item["id"] for item in json.loads(report["context"])["documents"]] == [
    "refund-summary", "refund-window"
]
assert {item["reason"] for item in report["skipped"]} == {"source_limit", "byte_budget"}
smaller = run("--bytes", "120")
assert [item["id"] for item in json.loads(smaller["context"])["documents"]] == ["refund-summary"]
print("245 bytes de orçamento:", report["bytes_used"], "usados")
print("120 bytes de orçamento:", smaller["bytes_used"], "usados")
print("Trechos descartados:", report["skipped"])

## Mude a cota e confira o limite do envelope

Mantenha 245 bytes e permita dois trechos por fonte. Depois reduza para o tamanho do envelope vazio. Compare o campo `context` em ambos os relatórios.

In [ ]:
with_history = run("--bytes", "245", "--per-source", "2")
document_ids = [item["id"] for item in json.loads(with_history["context"])["documents"]]
assert document_ids == ["refund-summary", "refund-history"]
print("Cota 2:", document_ids, "em", with_history["bytes_used"], "bytes")

empty = run("--bytes", "16")
assert json.loads(empty["context"]) == {"documents": []}
assert empty["bytes_used"] == 16
print("Orçamento mínimo:", empty["context"])